# STRING Database — Protein-Protein Interaction Network Analysis

**STRING** (Search Tool for the Retrieval of Interacting Genes/Proteins) is a database of known and predicted protein-protein interactions. Interactions include both direct (physical) and indirect (functional) associations. The database currently covers **67.6 million proteins** from **14,094 organisms**.

Each interaction is scored across multiple evidence channels:
| Channel | Description |
|---|---|
| `neighborhood` | Gene co-localization on chromosomes |
| `fusion` | Gene fusion events across species |
| `cooccurence` | Phylogenetic co-occurrence |
| `coexpression` | Gene expression correlation |
| `experimental` | Experimental interaction data (yeast-2-hybrid, co-IP, etc.) |
| `database` | Curated pathway/interaction databases |
| `textmining` | Co-mention in PubMed abstracts |
| `combined_score` | Integrated score (0–1000) |

**Reference:** Szklarczyk et al. (2023), *Nucleic Acids Research*, STRING v12.0

In [1]:
import requests
import time
import gzip
import io
from pathlib import Path

import polars as pl
import pandas as pd

# TODO

* [x] **Ingest data**
    * [x] Connect to STRING API (v12.0) and explore available endpoints
    * [x] Download protein-protein interaction (PPI) network for a model organism (e.g. *H. sapiens*)
    * [x] Parse interaction data (protein pairs, combined scores, channel-specific scores)
    * [x] Load into a Polars/Pandas DataFrame with appropriate dtypes
    * [x] Fetch protein metadata (preferred names, annotations, functional descriptions)
* [ ] **Explore and clean**
    * [ ] Summarize dataset dimensions, score distributions, and missing values
    * [ ] Filter interactions by combined score threshold (e.g. >= 700 for high confidence)
    * [ ] Examine the individual evidence channels (experimental, coexpression, text-mining, etc.)
* [ ] **Network analysis**
    * [ ] Build a NetworkX graph from the filtered interaction set
    * [ ] Compute basic network statistics (degree distribution, clustering coefficient, connected components)
    * [ ] Identify hub proteins (highest degree / betweenness centrality)
    * [ ] Detect communities using Louvain or Leiden clustering
* [ ] **Functional enrichment**
    * [ ] Retrieve GO term and KEGG pathway enrichments via STRING API for a gene set of interest
    * [ ] Visualize top enriched terms (bar/dot plots with Bokeh or Seaborn)
* [ ] **Visualization**
    * [ ] Plot score distributions across evidence channels
    * [ ] Visualize a subnetwork around a protein of interest (e.g. TP53)
    * [ ] Color nodes by community membership or functional annotation
* [ ] **Statistical analysis**
    * [ ] Compare observed network properties to random networks (degree-preserving null model)
    * [ ] Test whether hub proteins are enriched for disease-associated genes
    * [ ] Discuss multiple hypothesis correction considerations

## 1. Ingest Data

### 1.1 Connect to STRING API and Explore Endpoints

In [2]:
STRING_BASE = "https://string-db.org/api"
SPECIES_HUMAN = 9606          # NCBI taxon ID for Homo sapiens
STRING_VERSION = "12.0"

def string_get(endpoint: str, params: dict) -> requests.Response:
    """
    Send a GET request to the STRING API with a mandatory 1-second delay
    between calls, as required by STRING's fair-use policy.

    Parameters
    ----------
    endpoint : str
        API path (e.g. "tsv/network").
    params : dict
        Query parameters.

    Returns
    -------
    requests.Response
        Raw response object; caller is responsible for parsing.
    """
    url = f"{STRING_BASE}/{endpoint}"
    params.setdefault("caller_identity", "elixir_of_life_notebook")
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    time.sleep(1)   # respect STRING's rate-limit guidance
    return resp

# Quick connectivity check: resolve a single well-known protein to its STRING ID
resp = string_get("tsv/get_string_ids", {"identifiers": "TP53", "species": SPECIES_HUMAN})
print(resp.text[:500])

queryIndex	stringId	ncbiTaxonId	taxonName	preferredName	annotation
0	9606.ENSP00000269305	9606	Homo sapiens	TP53	Cellular tumor antigen p53; Acts as a tumor suppressor in many tumor types; induces growth arrest or apoptosis depending on the physiological circumstances and cell type. Involved in cell cycle regulation as a trans-activator that acts to negatively regulate cell division by controlling a set of genes required for this process. One of the activated genes is an inhibitor of cyclin-depe


### 1.2 Download Human PPI Network (Bulk File)

In [5]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Detailed links file includes per-channel subscores (~133 MB compressed)
LINKS_URL = (
    f"https://stringdb-downloads.org/download/"
    f"protein.links.detailed.v{STRING_VERSION}/"
    f"{SPECIES_HUMAN}.protein.links.detailed.v{STRING_VERSION}.txt.gz"
)
LINKS_PATH = DATA_DIR / f"{SPECIES_HUMAN}.protein.links.detailed.v{STRING_VERSION}.txt.gz"

if not LINKS_PATH.exists():
    print(f"Downloading {LINKS_PATH.name} ...")
    with requests.get(LINKS_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(LINKS_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {LINKS_PATH}")
else:
    print(f"Already downloaded: {LINKS_PATH}")

  139.6 / 139.6 MB
Saved to data/9606.protein.links.detailed.v12.0.txt.gz


### 1.3 Parse Interaction Data into a DataFrame

#### PPI DataFrame columns

| Column | Type | Description |
|---|---|---|
| `protein_a` / `protein_b` | `str` | STRING identifiers for the two interacting proteins, in the format `9606.ENSP00000XXXXXX` — the NCBI taxon ID followed by the Ensembl peptide ID |
| `neighborhood` | `UInt16` | Score based on **gene neighbourhood**: proteins whose genes are consistently co-located on chromosomes across many genomes tend to be functionally linked |
| `fusion` | `UInt16` | Score from **gene fusion events**: if two genes in one organism appear fused into one in another, their products likely interact |
| `cooccurence` | `UInt16` | **Phylogenetic co-occurrence**: genes that are either both present or both absent across a large panel of genomes are likely co-functional |
| `coexpression` | `UInt16` | Score from **correlated mRNA expression** across conditions/tissues — co-expressed genes are often co-functional |
| `experimental` | `UInt16` | Score from **direct experimental evidence** in *H. sapiens*: yeast two-hybrid, affinity purification–MS (AP-MS), co-immunoprecipitation, etc. |
| `database` | `UInt16` | Score from **manually curated pathway and interaction databases** (e.g. KEGG, Reactome, IntAct) — the most precise source but narrowest coverage |
| `textmining` | `UInt16` | Score from **co-mention in PubMed abstracts** — proteins frequently discussed together in the literature are assigned a positive association |
| `combined_score` | `UInt16` | **Integrated confidence score (0–1000)** combining all channels above via a probabilistic framework that corrects for the prior probability of any two proteins interacting by chance. Common thresholds: ≥ 400 (medium confidence), ≥ 700 (high), ≥ 900 (very high) |

> **Note:** The `detailed` file also contains `_transferred` variants of some channels (e.g. `experiments_transferred`), which propagate evidence from other organisms via orthology mapping — useful signal, but less direct than human-derived scores. These may be present in additional columns not shown in the truncated `head()` output above.

In [6]:
SCORE_COLS = [
    "neighborhood", "neighborhood_transferred",
    "fusion", "cooccurence",
    "coexpression", "coexpression_transferred",
    "experiments", "experiments_transferred",
    "database", "database_transferred",
    "textmining", "textmining_transferred",
    "combined_score",
]

# Polars reads directly from gzip — no manual decompression needed
ppi = pl.read_csv(
    LINKS_PATH,
    separator=" ",
    schema_overrides={col: pl.UInt16 for col in SCORE_COLS},  # scores fit in 0–1000
)

# Rename STRING's column headers to match our terminology
ppi = ppi.rename({"protein1": "protein_a", "protein2": "protein_b"})

print(f"Shape: {ppi.shape}")
print(f"Memory: {ppi.estimated_size('mb'):.1f} MB")
ppi.head(5)

Shape: (13715404, 10)
Memory: 811.0 MB


protein_a,protein_b,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
str,str,u16,u16,u16,u16,i64,u16,u16,u16
"""9606.ENSP00000000233""","""9606.ENSP00000356607""",0,0,0,45,134,0,81,173
"""9606.ENSP00000000233""","""9606.ENSP00000427567""",0,0,0,0,128,0,70,154
"""9606.ENSP00000000233""","""9606.ENSP00000253413""",0,0,0,118,49,0,69,151
"""9606.ENSP00000000233""","""9606.ENSP00000493357""",0,0,0,56,53,0,457,471
"""9606.ENSP00000000233""","""9606.ENSP00000324127""",0,0,0,0,46,0,197,201


### 1.4 Fetch Protein Metadata

In [7]:
INFO_URL = (
    f"https://stringdb-downloads.org/download/"
    f"protein.info.v{STRING_VERSION}/"
    f"{SPECIES_HUMAN}.protein.info.v{STRING_VERSION}.txt.gz"
)
INFO_PATH = DATA_DIR / f"{SPECIES_HUMAN}.protein.info.v{STRING_VERSION}.txt.gz"

if not INFO_PATH.exists():
    print(f"Downloading {INFO_PATH.name} ...")
    with requests.get(INFO_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(INFO_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved to {INFO_PATH}")
else:
    print(f"Already downloaded: {INFO_PATH}")

protein_info = pl.read_csv(INFO_PATH, separator="\t")

# Normalise column names: lowercase and replace spaces with underscores
protein_info.columns = [c.lower().replace(" ", "_") for c in protein_info.columns]

print(f"Shape: {protein_info.shape}")
protein_info.head(5)

Saved to data/9606.protein.info.v12.0.txt.gz
Shape: (19699, 4)


#string_protein_id,preferred_name,protein_size,annotation
str,str,i64,str
"""9606.ENSP00000000233""","""ARF5""",180,"""ADP-ribosylation factor 5; GTP…"
"""9606.ENSP00000000412""","""M6PR""",277,"""Cation-dependent mannose-6-pho…"
"""9606.ENSP00000001008""","""FKBP4""",459,"""Peptidyl-prolyl cis-trans isom…"
"""9606.ENSP00000001146""","""CYP26B1""",512,"""Cytochrome P450 26B1; Involved…"
"""9606.ENSP00000002125""","""NDUFAF7""",441,"""Protein arginine methyltransfe…"
